# CONQRR (2022)
---
[[paper]](https://arxiv.org/pdf/2206.01428)<br>CONQRR = CONtextualized Query Rewriting for Reranking

CONQRR — это подход, предложенный исследователями Google, который улучшает стадию **реранкинга** (reranking) в двухступенчатых системах информационного поиска за счет **контекстуального переписывания запроса** (contextualized query rewriting) для каждого документа.

### Контекст
В большинстве современных систем информационного поиска используется двухступенчатый подход: сначала быстрый **ретривер** (retriever) (например, BM25 или Dense Retrieval модель) отбирает широкий набор потенциально релевантных документов, а затем более сложный и вычислительно дорогой **реранкер** (reranker) уточняет порядок этих документов. Часто исходный запрос пользователя бывает коротким, неоднозначным или недостаточно полным для того, чтобы ретривер отобрал идеальный набор кандидатов. Даже если ретривер справился хорошо, реранкеру порой не хватает информации, чтобы точно оценить релевантность документа, особенно когда между запросом и документом существует **лексический разрыв** (lexical gap) или **семантическое несоответствие** (semantic mismatch).

### Идея
Основная идея CONQRR заключается в том, чтобы **динамически переписывать (обогащать) исходный запрос для каждого кандидата-документа**, а не использовать один и тот же запрос для всех документов на стадии реранкинга. Вместо того, чтобы просто подавать на вход реранкеру пару `(оригинальный_запрос, документ)`, CONQRR генерирует **новый, более детализированный и контекстуально обогащенный запрос** `rewritten_query_i` для каждого документа `document_i` из набора кандидатов. Этот `rewritten_query_i` затем используется вместе с `document_i` для оценки релевантности реранкером. Это позволяет реранкеру получить более точную и богатую информацию о том, как запрос связан с конкретным документом.

### Задача
CONQRR решает задачу **повышения точности реранкинга** в двухступенчатых системах информационного поиска. Цель состоит в том, чтобы, получив от ретривера набор из N документов-кандидатов для данного запроса, более эффективно отсортировать их по релевантности, выявляя наиболее подходящие.

### Альтернативы
До появления CONQRR существовали следующие подходы:
*   **Query Expansion (до ретривала/реранкинга):** Методы расширения запроса (например, с использованием псевдорелевантной обратной связи, WordNet, или нейронных моделей вроде Query2Doc [2021]) генерируют один расширенный запрос *до* выполнения ретривала или реранкинга.
    *   *Отличие:* CONQRR переписывает запрос *для каждого документа отдельно*, а не один раз глобально. Это позволяет создать документ-специфичный запрос, который учитывает уникальное содержание каждого документа.
*   **Стандартные реранкеры:** Модели на основе BERT (например, MonoBERT [2019], ColBERT [2020], T5-based rerankers), которые напрямую оценивают релевантность пары `(оригинальный_запрос, документ)`.
    *   *Отличие:* Эти реранкеры работают с исходным запросом без его модификации на основе содержимого документа. CONQRR вводит дополнительный шаг переписывания запроса, который позволяет реранкеру получить более точную входную информацию.
*   **Query Rewriting в диалоговых системах:** Методы, которые переписывают запросы для поддержания контекста в многоходовых диалогах (например, CONVRT [2020]).
    *   *Отличие:* Эти методы фокусируются на диалоговом контексте, тогда как CONQRR ориентирован на конкретный документ в списке кандидатов.

### Архитектура
Архитектура CONQRR состоит из двух основных компонентов, которые работают последовательно:

1.  **Модуль переписывания запроса (Query Rewriting Module, QRM):** Это генеративная модель, обычно основанная на архитектуре Transformer (например, T5), которая на вход принимает **оригинальный запрос пользователя** и **конкретный документ-кандидат**. Её задача — сгенерировать **контекстуально переписанный запрос** (`rewritten_query`), который наилучшим образом отражает связь между оригинальным запросом и данным документом.
2.  **Реранкер (Reranker):** Это стандартная модель кросс-энкодера (например, BERT-based Transformer), которая на вход принимает пару `(переписанный_запрос, документ)` и выдает **оценку релевантности** (relevance score). Этот реранкер является конечным оценщиком релевантности.

### Обучение
Процесс обучения CONQRR обычно состоит из нескольких этапов:

1.  **Предобучение QRM (опционально):** Модуль QRM может быть предварительно обучен на большом объеме данных для задачи перефразирования или генерации текста. Также его можно дообучить на синтетических данных, где запрос перефразируется с учетом контекста документа.
2.  **Обучение Реранкера:** Реранкер обучается в стандартной **супервизируемой манере** на наборах данных для ранжирования (например, MS MARCO). Он учится присваивать высокие оценки релевантным парам `(запрос, документ)` и низкие — нерелевантным. На этом этапе в качестве запроса используется *оригинальный* запрос.
3.  **Обучение QRM с использованием Reinforcement Learning (RL):** Этот этап является ключевым.
    *   QRM обучается генерировать переписанные запросы, которые *улучшают производительность уже обученного реранкера*.
    *   Реранкер здесь выступает в роли **функции вознаграждения** (reward function).
    *   Для каждой сгенерированной QRM пары `(rewritten_query, document)`, реранкер вычисляет оценку релевантности. Разница между этой оценкой и оценкой, полученной с *оригинальным* запросом, может использоваться как **вознаграждение** (reward) для QRM.
    *   QRM оптимизируется с использованием **методов градиентной политики** (policy gradient methods) (например, REINFORCE) для максимизации этого вознаграждения. Таким образом, QRM учится генерировать запросы, которые делают оценку релевантности реранкером более точной.

### Инференс
На этапе инференса (работы) CONQRR выполняет следующие шаги:

1.  **Инициальный ретривал:** Получение N документов-кандидатов от ретривера для исходного запроса `Q`.
2.  **Переписывание запроса:** Для *каждого* документа `D_i` из полученного набора:
    *   Модуль QRM принимает на вход `(Q, D_i)`.
    *   QRM генерирует **документ-специфичный переписанный запрос** `Q_i_rewritten`.
3.  **Реранкинг:** Для каждой пары `(Q_i_rewritten, D_i)`:
    *   Реранкер вычисляет **оценку релевантности** `S_i`.
4.  **Сортировка и выдача результатов:** Документы `D_i` сортируются в порядке убывания их оценок `S_i`, и возвращается топ-K наиболее релевантных документов.

### Результаты
В оригинальной статье авторы продемонстрировали, что CONQRR значительно улучшает метрики ранжирования по сравнению с сильными baseline-моделями, использующими только оригинальный запрос для реранкинга. Например:
*   На датасете **MS MARCO Passage Ranking**, CONQRR показал увеличение **nDCG@10** на 2-3 процентных пункта (pp) и **MRR@10** на 1-2 pp по сравнению с state-of-the-art T5-based реранкерами.
*   Особенно заметные улучшения наблюдались на сложных или неоднозначных запросах, где добавление контекста из документа существенно помогало реранкеру принимать более точные решения о релевантности.

## 📝 Критический анализ

```markdown
# CONQRR (2022)
---
[[paper]](https://arxiv.org/pdf/2206.01428)<br>CONQRR = CONtextualized Query Rewriting for Reranking

CONQRR — подход от Google для улучшения **реранкинга** в системах информационного поиска через **контекстуальное переписывание запроса** для каждого документа.

### Контекст
Современные системы поиска используют двухступенчатый подход: **ретривер** отбирает релевантные документы, а **реранкер** уточняет их порядок. Часто запросы пользователей короткие и неоднозначные, что затрудняет точную оценку релевантности.

### Идея
CONQRR динамически переписывает запрос для каждого документа, создавая **контекстуально обогащенный запрос** `rewritten_query_i`, который используется для оценки релевантности.

### Задача
CONQRR улучшает точность реранкинга, эффективно сортируя документы по релевантности.

### Альтернативы
- **Query Expansion:** Генерирует один расширенный запрос до ретривала.
- **Стандартные реранкеры:** Оценивают `(оригинальный_запрос, документ)` без модификации запроса.
- **Query Rewriting в диалогах:** Переписывает запросы для диалогового контекста.

### Архитектура
1. **Query Rewriting Module (QRM):** Генеративная модель (например, T5), создающая **переписанный запрос** для каждого документа.
2. **Реранкер:** Модель кросс-энкодера, оценивающая `(переписанный_запрос, документ)`.

### Обучение
1. **Предобучение QRM:** На данных для перефразирования.
2. **Обучение Реранкера:** На данных для ранжирования.
3. **Обучение QRM с RL:** Оптимизация переписывания для улучшения реранкера.

### Инференс
1. **Инициальный ретривал:** Получение N документов.
2. **Переписывание запроса:** Генерация `Q_i_rewritten` для каждого документа.
3. **Реранкинг:** Оценка релевантности `S_i`.
4. **Сортировка:** Возврат топ-K документов.

### Результаты
CONQRR улучшает метрики ранжирования на **MS MARCO Passage Ranking**, увеличивая **nDCG@10** на 2-3 pp и **MRR@10** на 1-2 pp по сравнению с T5-based реранкерами, особенно на сложных запросах.

<img src="img/img.png" width=500>
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Импортируем необходимые библиотеки
from transformers import T5Tokenizer, T5ForConditionalGeneration, BertTokenizer, BertForSequenceClassification
import torch

# Инициализация моделей и токенайзеров
# T5 для переписывания запроса
t5_tokenizer = T5Tokenizer.from_pretrained('t5-small')
t5_model = T5ForConditionalGeneration.from_pretrained('t5-small')

# BERT для реранкинга
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

# Пример исходного запроса и документов-кандидатов
original_query = "What is the capital of France?"
documents = [
    "Paris is the capital city of France, known for its art, fashion, and culture.",
    "France is a country in Europe with a rich history and diverse culture.",
    "The Eiffel Tower is one of the most famous landmarks in Paris, France."
]

# Функция для переписывания запроса с использованием T5
def rewrite_query(query, document):
    input_text = f"rewrite: {query} context: {document}"
    input_ids = t5_tokenizer.encode(input_text, return_tensors='pt')
    outputs = t5_model.generate(input_ids, max_length=50, num_beams=5, early_stopping=True)
    rewritten_query = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return rewritten_query

# Функция для оценки релевантности с использованием BERT
def rerank(rewritten_query, document):
    inputs = bert_tokenizer.encode_plus(rewritten_query, document, return_tensors='pt')
    outputs = bert_model(**inputs)
    relevance_score = torch.softmax(outputs.logits, dim=1)[0][1].item()  # Предполагаем, что класс 1 - релевантность
    return relevance_score

# Переписывание запроса и реранкинг для каждого документа
rewritten_queries = []
relevance_scores = []

for doc in documents:
    rewritten_query = rewrite_query(original_query, doc)
    rewritten_queries.append(rewritten_query)
    score = rerank(rewritten_query, doc)
    relevance_scores.append(score)

# Сортировка документов по релевантности
sorted_docs = sorted(zip(documents, rewritten_queries, relevance_scores), key=lambda x: x[2], reverse=True)

# Вывод результатов
for i, (doc, rewritten_query, score) in enumerate(sorted_docs):
    print(f"Document {i+1}: {doc}")
    print(f"Rewritten Query: {rewritten_query}")
    print(f"Relevance Score: {score:.4f}\n")
```

### Комментарии к коду:

1. **Переписывание запроса**: Используется модель T5 для генерации переписанного запроса. На вход подается комбинация исходного запроса и текста документа. Это позволяет создать контекстуально обогащенный запрос, специфичный для каждого документа.

2. **Реранкинг**: Используется модель BERT для оценки релевантности пары `(переписанный_запрос, документ)`. Модель возвращает оценку релевантности, которая используется для сортировки документов.

3. **Сортировка и вывод**: Документы сортируются по убыванию оценок релевантности, и выводятся в порядке их релевантности.

Этот код иллюстрирует основную идею CONQRR: динамическое переписывание запроса для каждого документа, что позволяет более точно оценивать релевантность на этапе реранкинга.